# Selection 2PCF Estimator Convergence (RSD-Style Plots)

This notebook reproduces the redshift-space-distortion-style convergence plots for the real-space 2PCF outputs under `data/halo_sampling_4_*`.

For each dataset family (model/sim/snapshot/selection/seeded-tag), it produces:
- curves by subvolume count
- absolute difference to a proxy full-box reference
- percentage difference to the same reference

with panel layout split by estimator mode:
- left: normal estimator
- right: corrected estimator (stored as `weighted` in these CSVs).

In [20]:
from pathlib import Path
import re
import sys
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / "data").is_dir() and (candidate / "src").is_dir()
    ),
    Path.cwd(),
)
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from galform_analysis.utils import setconfig
setconfig()

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

DATA_ROOT = REPO_ROOT / "data"
HALO4_PREFIX = "halo_sampling_4_"
CSV_ROOTS = sorted(
    path for path in DATA_ROOT.iterdir()
    if path.is_dir() and path.name.startswith(HALO4_PREFIX)
)
if not CSV_ROOTS:
    raise RuntimeError(f"No directories matching '{HALO4_PREFIX}*' found under {DATA_ROOT}")

PLOT_ROOT = REPO_ROOT / "examples" / "subvolume_statistics" / "_plots" / "selection_2pcf_estimator_convergence"
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_VALUES = {"lc16", "gp14"}
MODE_VALUES = {"normal", "weighted"}
MODE_LABELS = {"normal": "Normal", "weighted": "Corrected"}
FILENAME_PATTERN = re.compile(
    r"^halo_sampling_convergence_(?P<mode>[^_]+)_(?P<sim>[^_]+)_iz(?P<iz>\d+)(?:_n(?P<n_subvol>\d+))?\.csv$"
)
EPS = 1e-12
FULLBOX_N = 1024
TARGET_SIMS = {"L800"}

# Set to a list of tuples to display only selected groups in notebook output.
# Tuple format:
# (root_group, model, sim, iz, selection_path, set_tag)
SHOW_GROUPS_IN_NOTEBOOK = []

## 1. Discover And Index All Convergence CSVs

This step scans all `data/halo_sampling_4_*` directories, parses metadata from path + filename, and creates a dataset index used by all downstream plots.

For now, the notebook filters to `sim == "L800"` only.

Seeded labeling follows the dataset root naming convention (for example roots containing `seeded`) and seed path components (for example `seed_271828`).

In [21]:
def find_convergence_csvs(roots=CSV_ROOTS):
    csvs = []
    for root in roots:
        csvs.extend(path for path in root.rglob("halo_sampling_convergence_*.csv") if path.is_file())
    return sorted(csvs)


def infer_metadata(csv_path):
    rel = csv_path.relative_to(DATA_ROOT)
    parts = rel.parts
    match = FILENAME_PATTERN.match(csv_path.name)
    if match is None:
        return None

    root_group = parts[0] if len(parts) > 0 else ""
    model_idx = next((idx for idx, part in enumerate(parts) if part in MODEL_VALUES), -1)
    if model_idx < 0:
        return None

    model = parts[model_idx]
    mode = match.group("mode").lower()
    if mode not in MODE_VALUES:
        return None

    n_from_name = match.group("n_subvol")
    n_from_name = int(n_from_name) if n_from_name is not None else np.nan

    n_from_path = np.nan
    n_subvol_idx = -1
    for idx, part in enumerate(parts):
        n_match = re.match(r"^nsubvol_(\d+)$", part)
        if n_match:
            n_from_path = int(n_match.group(1))
            n_subvol_idx = idx
            break

    if np.isnan(n_from_name):
        n_hint = n_from_path
    elif np.isnan(n_from_path):
        n_hint = n_from_name
    else:
        n_hint = n_from_name if int(n_from_name) == int(n_from_path) else n_from_path

    selection_parts = []
    if model_idx > 1:
        selection_parts.extend(parts[1:model_idx])

    tail_end = n_subvol_idx if n_subvol_idx >= 0 else (len(parts) - 1)
    for part in parts[model_idx + 1 : tail_end]:
        if re.match(r"^iz\d+$", part):
            continue
        selection_parts.append(part)
    selection_path = "/".join(selection_parts)

    set_tag = "seeded" if ("seeded" in root_group.lower() or any(part.startswith("seed_") for part in parts)) else "unseeded"

    return {
        "csv_path": csv_path,
        "csv_relpath": str(rel),
        "root_group": root_group,
        "model": model,
        "mode": mode,
        "sim": match.group("sim"),
        "iz": int(match.group("iz")),
        "n_subvol_from_name": n_hint,
        "selection_path": selection_path,
        "set_tag": set_tag,
    }


def discover_records():
    rows = []
    for csv_path in find_convergence_csvs():
        meta = infer_metadata(csv_path)
        if meta is not None:
            rows.append(meta)

    records = pd.DataFrame(rows)
    if records.empty:
        pinned_roots = [path.name for path in CSV_ROOTS]
        raise RuntimeError(f"No convergence CSV files found under pinned halo_sampling_4 roots: {pinned_roots}")

    records = records.loc[records["sim"].isin(TARGET_SIMS)].copy()
    if records.empty:
        raise RuntimeError(f"No convergence CSV files found for target simulations: {sorted(TARGET_SIMS)}")

    records["group_key"] = records.apply(
        lambda row: "|".join(
            [
                str(row["root_group"]),
                str(row["model"]),
                str(row["sim"]),
                f"iz{int(row['iz'])}",
                str(row["selection_path"]),
                str(row["set_tag"]),
            ]
        ),
        axis=1,
    )
    return records


records_df = discover_records()
display(
    records_df[["root_group", "model", "sim", "iz", "selection_path", "set_tag", "mode", "n_subvol_from_name", "csv_relpath"]]
    .sort_values(["root_group", "model", "sim", "iz", "selection_path", "set_tag", "mode", "csv_relpath"])
    .head(80)
)

print(f"Pinned halo_sampling_4 source directories: {[path.name for path in CSV_ROOTS]}")
print(f"Target simulations: {sorted(TARGET_SIMS)}")
print(f"Discovered {len(records_df)} convergence CSV files across pinned halo_sampling_4 directories")
print(f"Dataset groups: {records_df['group_key'].nunique()}")
print(f"Plot output directory: {PLOT_ROOT}")

,root_group,model,sim,iz,selection_path,set_tag,mode,n_subvol_from_name,csv_relpath
0,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,1,halo_sampling_4_subvol_weighted/lc16/iz155/nto...
1,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,10,halo_sampling_4_subvol_weighted/lc16/iz155/nto...
2,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,100,halo_sampling_4_subvol_weighted/lc16/iz155/nto...
3,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,1024,halo_sampling_4_subvol_weighted/lc16/iz155/nto...
4,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,15,halo_sampling_4_subvol_weighted/lc16/iz155/nto...
...,...,...,...,...,...,...,...,...,...
75,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,8,halo_sampling_4_subvol_weighted/lc16/iz207/nto...
76,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,800,halo_sampling_4_subvol_weighted/lc16/iz207/nto...
77,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,weighted,1,halo_sampling_4_subvol_weighted/lc16/iz207/nto...
78,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,weighted,10,halo_sampling_4_subvol_weighted/lc16/iz207/nto...


Pinned halo_sampling_4 source directories: ['halo_sampling_4_subvol_weighted', 'halo_sampling_4_subvol_weighted_random', 'halo_sampling_4_subvol_weighted_random_seeded', 'halo_sampling_4_subvol_weighted_random_seeded_extremes']
Target simulations: ['L800']
Discovered 1039 convergence CSV files across pinned halo_sampling_4 directories
Dataset groups: 66
Plot output directory: /cosma/apps/durham/dc-hick2/galform_analysis/examples/subvolume_statistics/_plots/selection_2pcf_estimator_convergence


## 2. RSD-Style Figure Sets For 2PCF

For each dataset group we generate three 1x2 figure sets using only $\xi(r)$:
- curves
- absolute difference to reference
- percentage difference to reference

Panel layout:
- left column: normal estimator
- right column: corrected estimator (`weighted` mode)

Reference selection per mode:
- use $n_{subvol}=1024$ as the full-box reference
- plot and compare only the other $n_{subvol}$ values against that fixed reference

In [26]:
import textwrap


def sanitize_for_filename(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("_")


def choose_reference(mode_tables):
    if not mode_tables:
        return None, None, None

    if FULLBOX_N not in mode_tables:
        return None, None, None

    return mode_tables[FULLBOX_N], f"n={FULLBOX_N} full-box reference", FULLBOX_N


def aligned_curves(run_df, ref_df):
    merged = pd.merge(
        run_df[["r", "xi"]],
        ref_df[["r", "xi"]],
        on="r",
        how="inner",
        suffixes=("_run", "_ref"),
    )
    if merged.empty:
        return None, None, None, None, None

    r = merged["r"].to_numpy()
    run_xi = merged["xi_run"].to_numpy()
    ref_xi = merged["xi_ref"].to_numpy()
    run_s2 = (r ** 2) * run_xi
    ref_s2 = (r ** 2) * ref_xi
    return r, run_xi, ref_xi, run_s2, ref_s2


def set_positive_or_symlog(ax, values, default_linthresh):
    if not values:
        return

    concat = np.concatenate([np.ravel(val) for val in values if val is not None])
    concat = concat[np.isfinite(concat)]
    if concat.size == 0:
        return

    if np.all(concat > 0):
        ax.set_yscale("log")
        return

    base = np.nanpercentile(np.abs(concat), 25)
    linthresh = max(default_linthresh, 0.1 * base if np.isfinite(base) and base > 0 else default_linthresh)
    ax.set_yscale("symlog", linthresh=linthresh)


def load_mode_tables(group_df):
    buckets = defaultdict(lambda: defaultdict(list))

    for _, meta in group_df.iterrows():
        try:
            raw = pd.read_csv(meta["csv_path"])
        except Exception as exc:
            print(f"Failed to read {meta['csv_path']}: {exc}")
            continue

        if raw.empty:
            continue

        raw.columns = [str(col).strip().lower() for col in raw.columns]
        if "r" not in raw.columns or "xi" not in raw.columns:
            continue

        work = raw.copy()
        work["mode"] = str(meta["mode"]).lower()

        if "n_subvol" in work.columns:
            nvals = pd.to_numeric(work["n_subvol"], errors="coerce")
        else:
            nvals = pd.Series(meta["n_subvol_from_name"], index=work.index, dtype="float64")

        work["n_subvol"] = nvals
        work["r"] = pd.to_numeric(work["r"], errors="coerce")
        work["xi"] = pd.to_numeric(work["xi"], errors="coerce")
        work = work.dropna(subset=["n_subvol", "r", "xi"])

        if work.empty:
            continue

        work["n_subvol"] = work["n_subvol"].round().astype(int)

        for n_sub, sub_df in work.groupby("n_subvol"):
            agg = (
                sub_df[["r", "xi"]]
                .groupby("r", as_index=False)["xi"]
                .median()
                .sort_values("r")
                .reset_index(drop=True)
            )
            if not agg.empty:
                buckets[meta["mode"]][int(n_sub)].append(agg)

    mode_tables = {mode: {} for mode in MODE_VALUES}
    for mode in MODE_VALUES:
        for n_sub, frames in buckets[mode].items():
            merged = pd.concat(frames, ignore_index=True)
            merged = (
                merged.groupby("r", as_index=False)["xi"]
                .median()
                .sort_values("r")
                .reset_index(drop=True)
            )
            mode_tables[mode][int(n_sub)] = merged

    return mode_tables


TITLE_FONTSIZE_SUPER = 11
TITLE_FONTSIZE_AXES = 9
TITLE_WRAP_WIDTH = 110


def format_group_title(root_group, model, sim, iz, selection_path, set_tag):
    selection_label = selection_path if str(selection_path).strip() else "all"
    raw_group_title = " | ".join([
        str(root_group),
        str(model),
        str(sim),
        f"iz{int(iz)}",
        str(selection_label),
        str(set_tag),
    ])
    wrapped_group_title = textwrap.fill(
        raw_group_title,
        width=TITLE_WRAP_WIDTH,
        break_long_words=False,
        break_on_hyphens=False,
    )
    return selection_label, wrapped_group_title


plot_outputs = []
summary_rows = []
displayed_group_count = 0

group_columns = ["root_group", "model", "sim", "iz", "selection_path", "set_tag"]

for group_id, group_df in records_df.groupby(group_columns, dropna=False):
    root_group, model, sim, iz, selection_path, set_tag = group_id
    mode_tables = load_mode_tables(group_df)

    normal_ref_df, normal_ref_label, normal_ref_n = choose_reference(mode_tables["normal"])
    weighted_ref_df, weighted_ref_label, weighted_ref_n = choose_reference(mode_tables["weighted"])

    normal_compare = [n for n in sorted(mode_tables["normal"].keys()) if n != FULLBOX_N]
    weighted_compare = [n for n in sorted(mode_tables["weighted"].keys()) if n != FULLBOX_N]

    normal_has_plot_data = (normal_ref_df is not None) and (len(normal_compare) > 0)
    weighted_has_plot_data = (weighted_ref_df is not None) and (len(weighted_compare) > 0)

    summary_rows.append(
        {
            "root_group": root_group,
            "model": model,
            "sim": sim,
            "iz": iz,
            "selection_path": selection_path,
            "set_tag": set_tag,
            "normal_has_fullbox": FULLBOX_N in mode_tables["normal"],
            "weighted_has_fullbox": FULLBOX_N in mode_tables["weighted"],
            "normal_has_plot_data": normal_has_plot_data,
            "weighted_has_plot_data": weighted_has_plot_data,
            "n_normal_compared": len(normal_compare),
            "n_weighted_compared": len(weighted_compare),
            "normal_ref": normal_ref_label,
            "weighted_ref": weighted_ref_label,
            "normal_ref_n": normal_ref_n,
            "weighted_ref_n": weighted_ref_n,
        }
    )

    # Skip this dataset entirely if neither mode has valid comparison data.
    if not (normal_has_plot_data or weighted_has_plot_data):
        continue

    active_modes = []
    if normal_has_plot_data:
        active_modes.append(("normal", mode_tables["normal"], normal_ref_df, normal_ref_label))
    if weighted_has_plot_data:
        active_modes.append(("weighted", mode_tables["weighted"], weighted_ref_df, weighted_ref_label))

    n_all = sorted(set().union(*[set(sorted(tables.keys())) - {FULLBOX_N} for _, tables, _, _ in active_modes]))
    color_map = {
        n_sub: color
        for n_sub, color in zip(n_all, plt.cm.viridis(np.linspace(0.1, 0.95, len(n_all))))
    }

    ncols = len(active_modes)
    fig_width = 14 if ncols == 2 else 9

    figs = []
    fig_raw, axes_raw = plt.subplots(1, ncols, figsize=(fig_width, 5), sharex=True)
    axes_raw = np.atleast_1d(axes_raw)
    figs.append(("curves", fig_raw, axes_raw))

    fig_abs, axes_abs = plt.subplots(1, ncols, figsize=(fig_width, 5), sharex=True)
    axes_abs = np.atleast_1d(axes_abs)
    figs.append(("absdiff", fig_abs, axes_abs))

    fig_pct, axes_pct = plt.subplots(1, ncols, figsize=(fig_width, 5), sharex=True)
    axes_pct = np.atleast_1d(axes_pct)
    figs.append(("pctdiff", fig_pct, axes_pct))

    for col, (mode, tables, ref_df, ref_label) in enumerate(active_modes):
        ax_raw = axes_raw[col]
        ax_abs = axes_abs[col]
        ax_pct = axes_pct[col]

        compare_n_values = [n_sub for n_sub in sorted(tables) if n_sub != FULLBOX_N]

        raw_xi_values = []

        for n_sub in compare_n_values:
            run_df = tables[n_sub]
            color = color_map[n_sub]
            label = f"n={n_sub}"

            r, run_xi, ref_xi, _, _ = aligned_curves(run_df, ref_df)
            if r is None:
                continue

            ax_raw.plot(r, run_xi, lw=2, color=color, alpha=0.9, label=label)
            ax_abs.plot(r, np.abs(run_xi - ref_xi), lw=2, color=color, alpha=0.9, label=label)

            pct_xi = 100.0 * (run_xi - ref_xi) / (np.abs(ref_xi) + EPS)
            ax_pct.plot(r, pct_xi, lw=2, color=color, alpha=0.9, label=label)

            raw_xi_values.append(run_xi)

        ref_r = ref_df["r"].to_numpy()
        ref_xi = ref_df["xi"].to_numpy()

        ax_raw.plot(ref_r, ref_xi, color="black", lw=3, linestyle="--", label=ref_label)
        raw_xi_values.append(ref_xi)

        for ax in [ax_raw, ax_abs, ax_pct]:
            ax.set_xscale("log")
            ax.grid(True, alpha=0.3)

        set_positive_or_symlog(ax_raw, raw_xi_values, default_linthresh=1e-3)

        mode_title = MODE_LABELS.get(mode, mode.title())
        ax_raw.set_title(f"{mode_title} xi(r) ({ref_label})", fontsize=TITLE_FONTSIZE_AXES, pad=6)
        ax_abs.set_title(f"{mode_title} |Delta xi|", fontsize=TITLE_FONTSIZE_AXES, pad=6)
        ax_pct.set_title(f"{mode_title} xi % diff", fontsize=TITLE_FONTSIZE_AXES, pad=6)

        ax_pct.axhline(0.0, color="black", lw=1.5, linestyle="--")

    axes_raw[0].set_ylabel("xi(r)")
    axes_abs[0].set_ylabel("|Delta xi|")
    axes_pct[0].set_ylabel("100 Delta/(|ref|+epsilon)")

    for ax in axes_raw:
        ax.set_xlabel("r [h^-1 Mpc]")
    for ax in axes_abs:
        ax.set_xlabel("r [h^-1 Mpc]")
    for ax in axes_pct:
        ax.set_xlabel("r [h^-1 Mpc]")

    for ax in list(axes_raw) + list(axes_abs) + list(axes_pct):
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(fontsize=8, ncol=2)

    selection_label, group_title = format_group_title(
        root_group,
        model,
        sim,
        iz,
        selection_path,
        set_tag,
    )

    fig_raw.suptitle(
        f"2PCF xi(r) curves by subvolume\n{group_title}",
        y=0.995,
        fontsize=TITLE_FONTSIZE_SUPER,
    )
    fig_abs.suptitle(
        f"2PCF xi(r) absolute difference to reference\n{group_title}",
        y=0.995,
        fontsize=TITLE_FONTSIZE_SUPER,
    )
    fig_pct.suptitle(
        f"2PCF xi(r) percentage difference to reference\n{group_title}",
        y=0.995,
        fontsize=TITLE_FONTSIZE_SUPER,
    )

    for _, fig, _ in figs:
        fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.90), pad=1.0)

    stem = sanitize_for_filename(
        f"{root_group}_{model}_{sim}_iz{iz}_{selection_label}_{set_tag}"
    )
    for tag, fig, _ in figs:
        outpath = PLOT_ROOT / f"2pcf_{stem}_{tag}.png"
        fig.savefig(outpath, bbox_inches="tight")
        plot_outputs.append(outpath)

    show_group = (SHOW_GROUPS_IN_NOTEBOOK is None) or (group_id in SHOW_GROUPS_IN_NOTEBOOK)
    if show_group:
        for _, fig, _ in figs:
            display(fig)
        displayed_group_count += 1

    for _, fig, _ in figs:
        plt.close(fig)

summary_df = pd.DataFrame(summary_rows).sort_values(["root_group", "model", "sim", "iz", "selection_path", "set_tag"])
display(summary_df)

dataset_summary_path = PLOT_ROOT / "selection_2pcf_dataset_summary.csv"
summary_df.to_csv(dataset_summary_path, index=False)

print(f"Saved {len(plot_outputs)} plot files to {PLOT_ROOT}")
print(f"Displayed plots for {displayed_group_count} dataset groups in notebook output")
print(f"Saved dataset summary table: {dataset_summary_path}")

,root_group,model,sim,iz,selection_path,set_tag,normal_has_fullbox,weighted_has_fullbox,normal_has_plot_data,weighted_has_plot_data,n_normal_compared,n_weighted_compared,normal_ref,weighted_ref,normal_ref_n,weighted_ref_n
0,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
1,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
2,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e9/centrals_0,unseeded,False,False,False,False,0,12,None,None,None,NaN
3,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e9/centrals_1,unseeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
4,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,halo_sampling_4_subvol_weighted_random_seeded_...,lc16,L800,271,ntotal_1024/mhalo1e11_largerR_randfull/mhalo_1...,seeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
62,halo_sampling_4_subvol_weighted_random_seeded_...,lc16,L800,271,ntotal_1024/mhalo1e13_randfull/mhalo_1e13/cent...,seeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
63,halo_sampling_4_subvol_weighted_random_seeded_...,lc16,L800,271,ntotal_1024/mhalo1e13_randfull/mhalo_1e13/cent...,seeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0
64,halo_sampling_4_subvol_weighted_random_seeded_...,lc16,L800,271,ntotal_1024/mhalo1e6_randfull/mhalo_1e6/centra...,seeded,False,True,False,True,0,14,None,n=1024 full-box reference,None,1024.0


Saved 171 plot files to /cosma/apps/durham/dc-hick2/galform_analysis/examples/subvolume_statistics/_plots/selection_2pcf_estimator_convergence
Displayed plots for 0 dataset groups in notebook output
Saved dataset summary table: /cosma/apps/durham/dc-hick2/galform_analysis/examples/subvolume_statistics/_plots/selection_2pcf_estimator_convergence/selection_2pcf_dataset_summary.csv


## 3. Convergence Metrics Tables

For each group/mode and each non-reference $n_{subvol}$ (that is, $n_{subvol} \neq 1024$), this section reports:
- median absolute difference in $\xi(r)$
- median absolute percentage difference in $\xi(r)$

All differences are computed relative to the fixed full-box reference at $n_{subvol}=1024$.

In [23]:
convergence_rows = []

for group_id, group_df in records_df.groupby(group_columns, dropna=False):
    root_group, model, sim, iz, selection_path, set_tag = group_id
    mode_tables = load_mode_tables(group_df)

    for mode in ["normal", "weighted"]:
        tables = mode_tables[mode]
        ref_df, ref_label, ref_n = choose_reference(tables)
        if ref_df is None:
            continue

        for n_subvol, run_df in sorted(tables.items()):
            if int(n_subvol) == FULLBOX_N:
                continue

            r, run_xi, ref_xi, _, _ = aligned_curves(run_df, ref_df)
            if r is None:
                continue

            abs_diff_xi = np.abs(run_xi - ref_xi)
            pct_diff_xi = 100.0 * (run_xi - ref_xi) / (np.abs(ref_xi) + EPS)

            convergence_rows.append(
                {
                    "root_group": root_group,
                    "model": model,
                    "sim": sim,
                    "iz": int(iz),
                    "selection_path": selection_path,
                    "set_tag": set_tag,
                    "mode": mode,
                    "mode_label": MODE_LABELS.get(mode, mode.title()),
                    "n_subvol": int(n_subvol),
                    "reference": ref_label,
                    "reference_n": int(ref_n),
                    "med_abs_diff_xi": float(np.nanmedian(abs_diff_xi)),
                    "med_abs_pct_diff_xi": float(np.nanmedian(np.abs(pct_diff_xi))),
                }
            )

convergence_df = pd.DataFrame(convergence_rows)
if convergence_df.empty:
    print("No convergence rows were generated.")
else:
    convergence_df = convergence_df.sort_values(
        ["root_group", "model", "sim", "iz", "selection_path", "set_tag", "mode", "n_subvol"]
    )

    display(convergence_df.head(40))

    summary_by_group = (
        convergence_df.groupby(["root_group", "model", "sim", "iz", "selection_path", "set_tag", "mode"], dropna=False)
        .agg(
            n_curves=("n_subvol", "nunique"),
            best_med_abs_pct_xi=("med_abs_pct_diff_xi", "min"),
            worst_med_abs_pct_xi=("med_abs_pct_diff_xi", "max"),
        )
        .reset_index()
        .sort_values(["root_group", "model", "sim", "iz", "selection_path", "set_tag", "mode"])
    )

    display(summary_by_group)

    conv_path = PLOT_ROOT / "selection_2pcf_convergence_metrics_all_sets.csv"
    summary_path = PLOT_ROOT / "selection_2pcf_convergence_summary_all_sets.csv"
    convergence_df.to_csv(conv_path, index=False)
    summary_by_group.to_csv(summary_path, index=False)

    print(f"Saved convergence metrics table: {conv_path}")
    print(f"Saved convergence summary table: {summary_path}")

,root_group,model,sim,iz,selection_path,set_tag,mode,mode_label,n_subvol,reference,reference_n,med_abs_diff_xi,med_abs_pct_diff_xi
0,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,2,n=1024 full-box reference,1024,1.578212,59.680053
1,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,4,n=1024 full-box reference,1024,0.341400,14.366768
2,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,8,n=1024 full-box reference,1024,0.299427,11.877193
3,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,10,n=1024 full-box reference,1024,0.170919,8.525219
4,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,15,n=1024 full-box reference,1024,0.280239,5.601895
5,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,20,n=1024 full-box reference,1024,0.145277,4.176012
6,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,25,n=1024 full-box reference,1024,0.069600,3.034561
7,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,30,n=1024 full-box reference,1024,0.067712,1.928609
8,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,50,n=1024 full-box reference,1024,0.088592,3.071523
9,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,Corrected,100,n=1024 full-box reference,1024,0.053428,1.308870


,root_group,model,sim,iz,selection_path,set_tag,mode,n_curves,best_med_abs_pct_xi,worst_med_abs_pct_xi
0,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,14,0.149467,59.680053
1,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,weighted,14,0.124927,64.383817
2,halo_sampling_4_subvol_weighted,lc16,L800,155,ntotal_1024/custom/mhalo_1e9/centrals_1,unseeded,weighted,14,0.023571,27.357678
3,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,14,0.212669,84.776067
4,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,weighted,14,0.047216,66.418155
5,halo_sampling_4_subvol_weighted,lc16,L800,207,ntotal_1024/custom/mhalo_1e9/centrals_1,unseeded,weighted,14,0.005862,23.990299
6,halo_sampling_4_subvol_weighted,lc16,L800,271,ntotal_1024/custom/mhalo_1e11/centrals_0,unseeded,weighted,14,0.244650,29.193925
7,halo_sampling_4_subvol_weighted,lc16,L800,271,ntotal_1024/custom/mhalo_1e11/centrals_1,unseeded,weighted,14,0.045003,18.724819
8,halo_sampling_4_subvol_weighted,lc16,L800,271,ntotal_1024/custom/mhalo_1e9/centrals_0,unseeded,weighted,14,0.158316,30.558675
9,halo_sampling_4_subvol_weighted,lc16,L800,271,ntotal_1024/custom/mhalo_1e9/centrals_1,unseeded,weighted,14,0.007767,2.283967


Saved convergence metrics table: /cosma/apps/durham/dc-hick2/galform_analysis/examples/subvolume_statistics/_plots/selection_2pcf_estimator_convergence/selection_2pcf_convergence_metrics_all_sets.csv
Saved convergence summary table: /cosma/apps/durham/dc-hick2/galform_analysis/examples/subvolume_statistics/_plots/selection_2pcf_estimator_convergence/selection_2pcf_convergence_summary_all_sets.csv
